# CV Transfer Learning - Kaggle Free

Chọn một free runtime. Notebook chạy thật. `cpu-mini` dùng FakeData để kiểm tra pipeline nhẹ; `gpu-free` dùng CIFAR10 subset. Cả hai cố tải pretrained ResNet18. Nếu weights không tải được, random-weight fallback chỉ xác nhận code chạy và không đạt gate transfer learning.

## Environment check

In [ ]:
import importlib.util
import platform
HAS_TORCH = importlib.util.find_spec('torch') is not None
assert HAS_TORCH, 'Install the pinned torch/torchvision pair for this runtime'
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PROFILE = 'gpu-free' if DEVICE == 'cuda' else 'cpu-mini'
print({'python': platform.python_version(), 'torch': torch.__version__, 'device': DEVICE, 'profile': PROFILE})

## Install/verify dependencies

Nếu import lỗi, cài torch/torchvision theo selector chính thức của PyTorch rồi restart runtime. Không cài lại khi runtime đã có bản tương thích.

## Configuration and seed

In [ ]:
import random
import numpy as np
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
RUN_EPOCHS=5 if PROFILE=='gpu-free' else 1  # On resume, this many additional epochs run.
SAMPLES=3000 if PROFILE=='gpu-free' else 160
BATCH_SIZE=32 if PROFILE=='gpu-free' else 8
IMAGE_SIZE=160 if PROFILE=='gpu-free' else 96
print({'seed':SEED,'run_epochs':RUN_EPOCHS,'samples':SAMPLES,'batch':BATCH_SIZE})

## Data acquisition with mini fallback

In [ ]:
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from torchvision.models import ResNet18_Weights
WEIGHTS=ResNet18_Weights.DEFAULT
mean,std=WEIGHTS.transforms().mean,WEIGHTS.transforms().std
def build_transforms(image_size):
    train=transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.Resize((image_size,image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean,std),
    ])
    evaluate=transforms.Compose([
        transforms.Resize((image_size,image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean,std),
    ])
    return train,evaluate
DATA_SOURCE='FakeData cpu-mini smoke'
if PROFILE=='cpu-mini':
    base_dataset=datasets.FakeData(size=SAMPLES,image_size=(3,IMAGE_SIZE,IMAGE_SIZE),num_classes=2,transform=None,random_offset=SEED)
    class_names=['class-0','class-1']
else:
    try:
        full=datasets.CIFAR10(root='data',train=True,download=True,transform=None)
        base_dataset=Subset(full,list(range(min(SAMPLES,len(full)))))
        class_names=full.classes
        DATA_SOURCE='CIFAR10 subset'
    except Exception as error:
        print(f'CIFAR10 unavailable ({type(error).__name__}); switching to a one-epoch FakeData smoke test. Do not report its accuracy as model quality.')
        PROFILE='cpu-mini-offline-fallback'
        RUN_EPOCHS=1; SAMPLES=160; BATCH_SIZE=8; IMAGE_SIZE=96
        base_dataset=datasets.FakeData(size=SAMPLES,image_size=(3,IMAGE_SIZE,IMAGE_SIZE),num_classes=2,transform=None,random_offset=SEED)
        class_names=['class-0','class-1']
        DATA_SOURCE='FakeData internet fallback smoke'
train_transform,eval_transform=build_transforms(IMAGE_SIZE)
class TransformSubset(Dataset):
    def __init__(self,dataset,indices,transform):
        self.dataset=dataset; self.indices=list(indices); self.transform=transform
    def __len__(self): return len(self.indices)
    def __getitem__(self,index):
        image,label=self.dataset[self.indices[index]]
        return self.transform(image),label
generator=torch.Generator().manual_seed(SEED)
indices=torch.randperm(len(base_dataset),generator=generator).tolist()
train_n=int(0.7*len(indices)); val_n=int(0.15*len(indices))
train_indices=indices[:train_n]; val_indices=indices[train_n:train_n+val_n]; test_indices=indices[train_n+val_n:]
train_ds=TransformSubset(base_dataset,train_indices,train_transform)
val_ds=TransformSubset(base_dataset,val_indices,eval_transform)
test_ds=TransformSubset(base_dataset,test_indices,eval_transform)
loaders={name:DataLoader(ds,batch_size=BATCH_SIZE,shuffle=name=='train',num_workers=0) for name,ds in [('train',train_ds),('validation',val_ds),('test',test_ds)]}
print({'source':DATA_SOURCE,'normalization':{'mean':mean,'std':std},'image_size':IMAGE_SIZE,'run_epochs':RUN_EPOCHS,'train_augmentation':'RandomHorizontalFlip',**{k:len(v.dataset) for k,v in loaders.items()}})

## Data validation

In [ ]:
assert min(len(loader.dataset) for loader in loaders.values())>0
images,labels=next(iter(loaders['train']))
assert images.ndim==4 and labels.ndim==1
NUM_CLASSES=len(class_names)
print({'shape':tuple(images.shape),'classes':NUM_CLASSES})

## Baseline

In [ ]:
from collections import Counter
train_labels=[]
for _,labels in loaders['train']: train_labels.extend(labels.tolist())
majority=max(Counter(train_labels).values())/len(train_labels)
print({'majority_accuracy':majority})

## Training

In [ ]:
from pathlib import Path
import copy
import shutil
from torchvision.models import resnet18
from torch import nn
PRETRAINED_BACKBONE=True
try:
    model=resnet18(weights=WEIGHTS)
except Exception as error:
    print(f'Pretrained weights unavailable ({type(error).__name__}); using random weights for execution-only fallback. This run does not meet the transfer-learning gate.')
    PRETRAINED_BACKBONE=False
    PROFILE='cpu-mini-offline-fallback'; DATA_SOURCE=f'{DATA_SOURCE}; random-weight execution fallback'
    model=resnet18(weights=None)
for parameter in model.parameters(): parameter.requires_grad=False
model.fc=nn.Linear(model.fc.in_features,len(class_names))
model=model.to(DEVICE)
NUM_CLASSES=len(class_names)
optimizer=torch.optim.Adam(model.fc.parameters(),lr=1e-3)
criterion=nn.CrossEntropyLoss(); best=float('inf'); patience=2; stale=0; history=[]
Path('artifacts').mkdir(exist_ok=True)
best_checkpoint_path=Path('artifacts/best_checkpoint.pt')
last_checkpoint_path=Path('artifacts/last_checkpoint.pt')
config={'profile':PROFILE,'samples':SAMPLES,'batch_size':BATCH_SIZE,'image_size':IMAGE_SIZE}
start_epoch=0
best_model_state=None
RESUME=False  # Upload last_checkpoint.pt or artifacts.zip, keep the same config/labels, then set True.
if RESUME:
    uploaded_checkpoint_path=Path('last_checkpoint.pt')
    uploaded_archive_path=Path('artifacts.zip')
    if not last_checkpoint_path.exists() and uploaded_checkpoint_path.exists():
        shutil.copy2(uploaded_checkpoint_path,last_checkpoint_path)
    if not last_checkpoint_path.exists() and uploaded_archive_path.exists():
        shutil.unpack_archive(uploaded_archive_path,Path('artifacts'))
    resume_checkpoint_path=last_checkpoint_path
    if not resume_checkpoint_path.exists():
        raise FileNotFoundError('RESUME=True but no artifacts/last_checkpoint.pt, uploaded last_checkpoint.pt, or artifacts.zip was found.')
    checkpoint=torch.load(resume_checkpoint_path,map_location=DEVICE,weights_only=True)
    assert checkpoint['config']==config
    assert checkpoint['class_names']==class_names
    assert checkpoint['pretrained_backbone']==PRETRAINED_BACKBONE
    model.load_state_dict(checkpoint['model']); optimizer.load_state_dict(checkpoint['optimizer'])
    start_epoch=int(checkpoint['epoch']); best=float(checkpoint['best_validation_loss'])
    history=list(checkpoint['history']); stale=int(checkpoint['stale_epochs'])
    best_model_state=checkpoint['best_model']
    torch.save(checkpoint,best_checkpoint_path)  # Recreate best checkpoint in this fresh runtime.
target_epoch=start_epoch+RUN_EPOCHS
for epoch in range(start_epoch,target_epoch):
    model.train(); train_loss=0.0
    for x,y in loaders['train']:
        x,y=x.to(DEVICE),y.to(DEVICE); optimizer.zero_grad(); loss=criterion(model(x),y); loss.backward(); optimizer.step(); train_loss+=loss.item()*len(x)
    model.eval(); val_loss=0.0
    with torch.no_grad():
        for x,y in loaders['validation']:
            x,y=x.to(DEVICE),y.to(DEVICE); val_loss+=criterion(model(x),y).item()*len(x)
    row={'epoch':epoch+1,'train_loss':train_loss/len(train_ds),'validation_loss':val_loss/len(val_ds)}; history.append(row); print(row)
    improved=row['validation_loss']<best
    if improved:
        best=row['validation_loss']; stale=0; best_model_state=copy.deepcopy(model.state_dict())
    else:
        stale+=1
    checkpoint={'model':model.state_dict(),'best_model':best_model_state,'optimizer':optimizer.state_dict(),'epoch':epoch+1,'best_validation_loss':best,'stale_epochs':stale,'history':history,'seed':SEED,'profile':PROFILE,'config':config,'class_names':class_names,'pretrained_backbone':PRETRAINED_BACKBONE}
    torch.save(checkpoint,last_checkpoint_path)
    if improved:
        torch.save(checkpoint,best_checkpoint_path)
    if stale>=patience: break
# Optional stretch: unfreeze layer4 after the frozen-head baseline, then train one controlled extra epoch.
UNFREEZE_LAST_BLOCK=False
fine_tune_history=[]
if UNFREEZE_LAST_BLOCK and PRETRAINED_BACKBONE and not PROFILE.startswith('cpu-mini'):
    for parameter in model.layer4.parameters(): parameter.requires_grad=True
    optimizer=torch.optim.Adam([{'params':model.layer4.parameters(),'lr':1e-5},{'params':model.fc.parameters(),'lr':1e-4}])
    model.train(); fine_tune_loss=0.0
    for x,y in loaders['train']:
        x,y=x.to(DEVICE),y.to(DEVICE); optimizer.zero_grad(); loss=criterion(model(x),y); loss.backward(); optimizer.step(); fine_tune_loss+=loss.item()*len(x)
    model.eval(); fine_tune_validation_loss=0.0
    with torch.no_grad():
        for x,y in loaders['validation']:
            x,y=x.to(DEVICE),y.to(DEVICE); fine_tune_validation_loss+=criterion(model(x),y).item()*len(x)
    fine_tune_history.append({'policy':'unfreeze-layer4','train_loss':fine_tune_loss/len(train_ds),'validation_loss':fine_tune_validation_loss/len(val_ds)})
    print({'frozen_head_best_validation_loss':best,'fine_tune':fine_tune_history[-1]})

## Evaluation and error analysis

In [ ]:
from PIL import Image,ImageDraw
assert best_checkpoint_path.exists(), 'Run at least one training epoch before evaluation.'
model.load_state_dict(torch.load(best_checkpoint_path,map_location=DEVICE,weights_only=True)['best_model']); model.eval()
truth=[]; predicted=[]; failure_candidates=[]
failure_dir=Path('artifacts/failure-images'); failure_dir.mkdir(parents=True,exist_ok=True)
def save_failure_image(tensor,path,expected,actual):
    mean_tensor=torch.tensor(mean).view(3,1,1); std_tensor=torch.tensor(std).view(3,1,1)
    pixels=(tensor.cpu()*std_tensor+mean_tensor).clamp(0,1)
    image=transforms.ToPILImage()(pixels)
    canvas=Image.new('RGB',(image.width,image.height+24),'white'); canvas.paste(image,(0,24))
    ImageDraw.Draw(canvas).text((4,4),f'truth={class_names[expected]} predicted={class_names[actual]}',fill='black')
    canvas.save(path)
with torch.no_grad():
    for batch,(x,y) in enumerate(loaders['test']):
        logits=model(x.to(DEVICE)); probabilities=torch.softmax(logits,dim=1).cpu(); p=probabilities.argmax(1)
        truth.extend(y.tolist()); predicted.extend(p.tolist())
        for i,(expected,actual) in enumerate(zip(y.tolist(),p.tolist())):
            if expected!=actual:
                confidence=float(probabilities[i,actual])
                failure_candidates.append({'id':f'batch-{batch}-item-{i}','truth':expected,'predicted':actual,'confidence':confidence,'tensor':x[i].cpu()})
# Review confident-wrong cases first. Replace error_type after inspecting each image; keep `unreviewed` explicit until then.
failure_candidates.sort(key=lambda item:item['confidence'],reverse=True)
failures=[]
for candidate in failure_candidates[:20]:
    image_path=f"artifacts/failure-images/{candidate['id']}.png"
    save_failure_image(candidate.pop('tensor'),image_path,candidate['truth'],candidate['predicted'])
    candidate.update({'image':image_path,'sampling_rule':'confident-wrong','error_type':'unreviewed'})
    failures.append(candidate)
from sklearn.metrics import accuracy_score,confusion_matrix,f1_score,precision_recall_fscore_support
precision,recall,f1,support=precision_recall_fscore_support(truth,predicted,labels=list(range(NUM_CLASSES)),zero_division=0)
confusion_counts=confusion_matrix(truth,predicted,labels=list(range(NUM_CLASSES)))
row_totals=confusion_counts.sum(axis=1,keepdims=True)
confusion_matrix_normalized=np.divide(confusion_counts,row_totals,out=np.zeros_like(confusion_counts,dtype=float),where=row_totals!=0)
metrics={'accuracy':accuracy_score(truth,predicted),'macro_f1':f1_score(truth,predicted,average='macro'),'weighted_f1':f1_score(truth,predicted,average='weighted'),'per_class':{str(i):{'precision':float(precision[i]),'recall':float(recall[i]),'f1':float(f1[i]),'support':int(support[i])} for i in range(NUM_CLASSES)},'confusion_matrix_counts':confusion_counts.tolist(),'confusion_matrix_normalized':confusion_matrix_normalized.tolist(),'failure_examples':failures}
assert len(list(failure_dir.glob('*.png')))==len(failures)<=20
failure_limitation=None if len(failures)==20 else f'Only {len(failures)} misclassifications existed; exported all instead of padding to 20.'
metrics['failure_evidence']={'exported':len(failures),'cap':20,'sampling_rule':'confident-wrong','taxonomy_status':'manual-review-required' if failures else 'no-misclassifications','limitation':failure_limitation}
print({'metrics':metrics,'failure_images':len(failures),'failure_dir':str(failure_dir),'limitation':failure_limitation})

## Save artifacts and manifest

In [ ]:
import hashlib
import json
import shutil
from pathlib import Path
Path('artifacts/metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
Path('artifacts/history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
best_checksum=hashlib.sha256(best_checkpoint_path.read_bytes()).hexdigest()
last_checksum=hashlib.sha256(last_checkpoint_path.read_bytes()).hexdigest()
manifest={'seed':SEED,'profile':PROFILE,'device':DEVICE,'data_source':DATA_SOURCE,'pretrained_backbone':PRETRAINED_BACKBONE,'transfer_learning_evidence':PRETRAINED_BACKBONE,'quality_evidence':DATA_SOURCE=='CIFAR10 subset' and PRETRAINED_BACKBONE,'epochs_completed':len(history),'best_checkpoint_sha256':best_checksum,'last_checkpoint_sha256':last_checksum,'classes':class_names}
Path('artifacts/manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
Path('artifacts/model-card.md').write_text('# Model card\n\nFrozen ResNet18 learning lab. Not for production. FakeData validates execution only. Random weights do not satisfy the transfer-learning gate. Model-quality evidence requires a real dataset and pretrained weights.\n',encoding='utf-8')
archive=shutil.make_archive('artifacts','zip',root_dir='artifacts')
assert Path(archive).is_file()
print({'download':archive,'bytes':Path(archive).stat().st_size,'files':sorted(p.name for p in Path('artifacts').iterdir())})

## Release runtime

Tải `artifacts.zip`. Colab: Runtime > Disconnect and delete runtime. Kaggle: Save Version, download output, tắt accelerator/session.